In [14]:
# ============================================================
# 0. LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path
import os
import re
import shutil
from decimal import Decimal

In [ ]:
# ============================================================
# 1. RUTAS Y CONFIGURACIÓN
# ============================================================

BASE_DIR = Path(r"D:\Descargas")

PROM_FOLDER = BASE_DIR / "prom_slurm_joined"
PROM_PARQUET_DIR = PROM_FOLDER / "prom_slurm_joined.parquet"

HARDWARE_PATH = BASE_DIR / "node_hardware_info.parquet"
SLURM_PATH = BASE_DIR / "slurm_table_cleaned.parquet"

# Archivo final único
OUTPUT_SINGLE_FILE = BASE_DIR / "datacenter_enriched_1gb_single.parquet"


# Tamaño objetivo aproximado del archivo final
TARGET_GB = 1.0
TARGET_BYTES = TARGET_GB * (1024 ** 3)

# Ajustes seguros para laptop de 16 GB RAM
BATCH_SIZE = 50_000

print("Existe carpeta prom_slurm_joined:", PROM_FOLDER.exists())
print("Existe carpeta prom_slurm_joined.parquet:", PROM_PARQUET_DIR.exists())
print("Existe node_hardware_info:", HARDWARE_PATH.exists())
print("Existe slurm_table_cleaned:", SLURM_PATH.exists())
print("Archivo final:", OUTPUT_SINGLE_FILE)

Existe carpeta prom_slurm_joined: True
Existe carpeta prom_slurm_joined.parquet: True
Existe node_hardware_info: True
Existe slurm_table_cleaned: True
Archivo final: D:\Descargas\datacenter_enriched_1gb_single.parquet


In [16]:
# ============================================================
# 2. BUSCAR ARCHIVOS PARQUET REALES DEL DATASET GRANDE
# ============================================================

parquet_parts = [
    p for p in PROM_FOLDER.rglob("*.parquet")
    if p.is_file() and not p.name.startswith(".")
]

parquet_parts = sorted(parquet_parts)

print("Número de archivos parquet reales encontrados:", len(parquet_parts))

print("\nPrimeros archivos:")
for file in parquet_parts[:10]:
    print(file)

Número de archivos parquet reales encontrados: 200

Primeros archivos:
D:\Descargas\prom_slurm_joined\prom_slurm_joined.parquet\part-00000-ed3d2dff-1a97-4ddb-ad80-88573fd4bf4d-c000.snappy.parquet
D:\Descargas\prom_slurm_joined\prom_slurm_joined.parquet\part-00001-ed3d2dff-1a97-4ddb-ad80-88573fd4bf4d-c000.snappy.parquet
D:\Descargas\prom_slurm_joined\prom_slurm_joined.parquet\part-00002-ed3d2dff-1a97-4ddb-ad80-88573fd4bf4d-c000.snappy.parquet
D:\Descargas\prom_slurm_joined\prom_slurm_joined.parquet\part-00003-ed3d2dff-1a97-4ddb-ad80-88573fd4bf4d-c000.snappy.parquet
D:\Descargas\prom_slurm_joined\prom_slurm_joined.parquet\part-00004-ed3d2dff-1a97-4ddb-ad80-88573fd4bf4d-c000.snappy.parquet
D:\Descargas\prom_slurm_joined\prom_slurm_joined.parquet\part-00005-ed3d2dff-1a97-4ddb-ad80-88573fd4bf4d-c000.snappy.parquet
D:\Descargas\prom_slurm_joined\prom_slurm_joined.parquet\part-00006-ed3d2dff-1a97-4ddb-ad80-88573fd4bf4d-c000.snappy.parquet
D:\Descargas\prom_slurm_joined\prom_slurm_joined.parqu

In [18]:
# ============================================================
# 3. LEER ESQUEMA DEL PRIMER ARCHIVO
# ============================================================

first_file = parquet_parts[0]
schema = pq.read_schema(first_file)
available_columns = schema.names

print("Primer archivo usado para esquema:")
print(first_file)

print("\nNúmero de columnas disponibles:", len(available_columns))

print("\nColumnas disponibles:")
for col in available_columns:
    print(col)

Primer archivo usado para esquema:
D:\Descargas\prom_slurm_joined\prom_slurm_joined.parquet\part-00000-ed3d2dff-1a97-4ddb-ad80-88573fd4bf4d-c000.snappy.parquet

Número de columnas disponibles: 101

Columnas disponibles:
prom_id
timestamp
node
node_time_seconds
node_load15
node_power_usage
up
node_netstat_Tcp_OutSegs
node_netstat_Tcp_InErrs
node_context_switches_total
node_load5
node_load1
node_memory_Active_bytes
node_netstat_Tcp_RetransSegs
node_netstat_Udp_InErrors
node_memory_Dirty_bytes
node_ambient_temp
node_netstat_Icmp_InMsgs
node_netstat_Udp_InDatagrams
node_intr_total
node_netstat_Tcp_InSegs
node_memory_Percpu_bytes
node_boot_time_seconds
node_netstat_Udp_OutDatagrams
node_netstat_Icmp_InErrors
node_procs_blocked
node_netstat_Icmp_OutMsgs
node_memory_MemFree_bytes
node_procs_running
node_forks_total
node_hwmon_temp_celsius_min
node_hwmon_temp_celsius_mean
node_hwmon_temp_celsius_max
node_filesystem_avail_bytes_sum
node_filesystem_files_sum
node_network_transmit_bytes_total_sum

In [19]:
# ============================================================
# 4. SELECCIONAR COLUMNAS ÚTILES PARA EL PROYECTO
# ============================================================

candidate_columns = [
    # Identificación y tiempo
    "prom_id",
    "timestamp",
    "timestamp_seconds",
    "timestamp_seconds_delta",
    "node",
    "gpu_node",

    # Carga CPU / sistema
    "node_load1",
    "node_load5",
    "node_load15",
    "node_procs_running",
    "node_procs_blocked",
    "node_power_usage",
    "up",

    # Energía / potencia CPU
    "node_rapl_package_joules_total_sum",
    "node_rapl_package_joules_total_sum_delta",
    "node_rapl_package_power_sum",

    # Memoria RAM
    "node_memory_Active_bytes",
    "node_memory_MemFree_bytes",
    "node_memory_Dirty_bytes",
    "node_memory_MemAvailable_bytes",
    "node_memory_MemTotal_bytes",

    # Temperatura del nodo
    "node_hwmon_temp_celsius_min",
    "node_hwmon_temp_celsius_mean",
    "node_hwmon_temp_celsius_max",
    "node_thermal_zone_temp_min",
    "node_thermal_zone_temp_mean",
    "node_thermal_zone_temp_max",

    # GPU
    "nvidia_gpu_memory_used_bytes_sum",
    "nvidia_gpu_temperature_celsius_0",
    "nvidia_gpu_temperature_celsius_1",
    "nvidia_gpu_temperature_celsius_2",
    "nvidia_gpu_temperature_celsius_3",
    "nvidia_gpu_temperature_celsius_min",
    "nvidia_gpu_temperature_celsius_mean",
    "nvidia_gpu_temperature_celsius_max",
    "nvidia_gpu_fanspeed_percent_min",
    "nvidia_gpu_fanspeed_percent_mean",
    "nvidia_gpu_fanspeed_percent_max",
    "nvidia_gpu_power_usage_milliwatts_0",
    "nvidia_gpu_power_usage_milliwatts_1",
    "nvidia_gpu_power_usage_milliwatts_2",
    "nvidia_gpu_power_usage_milliwatts_3",
    "nvidia_gpu_power_usage_milliwatts_min",
    "nvidia_gpu_power_usage_milliwatts_mean",
    "nvidia_gpu_power_usage_milliwatts_max",
    "nvidia_gpu_power_usage_milliwatts_sum",
    "nvidia_gpu_duty_cycle_min",
    "nvidia_gpu_duty_cycle_mean",
    "nvidia_gpu_duty_cycle_max",

    # Disco
    "node_disk_read_bytes_total_sum_delta",
    "node_disk_written_bytes_total_sum_delta",
    "node_disk_reads_completed_total_sum_delta",
    "node_disk_writes_completed_total_sum_delta",
    "node_disk_io_now_sum",

    # Red
    "node_network_receive_bytes_total_sum_delta",
    "node_network_transmit_bytes_total_sum_delta",
    "node_network_receive_packets_total_sum_delta",
    "node_network_transmit_packets_total_sum_delta",

    # SLURM / jobs
    "slurm_id",
    "start_date",
    "end_date",
    "nodetypes",
    "numnodes",
    "numcores",
    "submit",
    "state",
    "__index_level_0__",
    "slurm_nodes"
]

columns_to_read = [col for col in candidate_columns if col in available_columns]

print("Columnas seleccionadas:", len(columns_to_read))

for col in columns_to_read:
    print(col)

Columnas seleccionadas: 65
prom_id
timestamp
timestamp_seconds
timestamp_seconds_delta
node
gpu_node
node_load1
node_load5
node_load15
node_procs_running
node_procs_blocked
node_power_usage
up
node_rapl_package_joules_total_sum
node_rapl_package_joules_total_sum_delta
node_rapl_package_power_sum
node_memory_Active_bytes
node_memory_MemFree_bytes
node_memory_Dirty_bytes
node_hwmon_temp_celsius_min
node_hwmon_temp_celsius_mean
node_hwmon_temp_celsius_max
node_thermal_zone_temp_min
node_thermal_zone_temp_mean
node_thermal_zone_temp_max
nvidia_gpu_memory_used_bytes_sum
nvidia_gpu_temperature_celsius_0
nvidia_gpu_temperature_celsius_1
nvidia_gpu_temperature_celsius_2
nvidia_gpu_temperature_celsius_3
nvidia_gpu_temperature_celsius_min
nvidia_gpu_temperature_celsius_mean
nvidia_gpu_temperature_celsius_max
nvidia_gpu_fanspeed_percent_min
nvidia_gpu_fanspeed_percent_mean
nvidia_gpu_fanspeed_percent_max
nvidia_gpu_power_usage_milliwatts_0
nvidia_gpu_power_usage_milliwatts_1
nvidia_gpu_power_usag

In [20]:
# ============================================================
# 5. FUNCIONES DE LIMPIEZA DE TIPOS
# ============================================================

def clean_decimal_columns_from_table(table):
    """
    Convierte columnas Decimal de PyArrow a float para evitar errores
    como: Decimal value does not fit in precision 8.
    """
    df = table.to_pandas()

    for field in table.schema:
        if pa.types.is_decimal(field.type):
            col = field.name
            if col in df.columns:
                df[col] = df[col].apply(
                    lambda x: float(x) if pd.notna(x) else np.nan
                )

    return df


def normalize_object_columns(df):
    """
    Convierte columnas object problemáticas a string.
    Evita errores de listas, objetos o decimales al guardar en parquet.
    """
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype("string")
    return df


def safe_to_numeric(df, columns):
    """
    Convierte columnas a numéricas si existen.
    Los textos se convierten en NaN.
    """
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

In [21]:
# ============================================================
# 6. CARGAR Y PREPARAR HARDWARE INFO
# ============================================================

df_hw_merge = None

if HARDWARE_PATH.exists():
    hw_table = pq.read_table(HARDWARE_PATH)
    df_hw = clean_decimal_columns_from_table(hw_table)

    print("Hardware info cargado:")
    print(df_hw.shape)

    print("\nColumnas de hardware:")
    print(df_hw.columns.tolist())

    # Buscar columnas de ubicación física si existen
    location_keywords = [
        "rack", "row", "room", "site", "location", "cabinet",
        "chassis", "slot", "position", "floor", "pod", "island"
    ]

    location_cols = [
        col for col in df_hw.columns
        if any(k in col.lower() for k in location_keywords)
    ]

    print("\nColumnas posibles de ubicación física:")
    print(location_cols)

    # Buscar llave para unir con el dataset principal
    possible_node_keys = ["node", "node_id", "hostname", "host", "name"]

    hw_key = None
    for key in possible_node_keys:
        if key in df_hw.columns:
            hw_key = key
            break

    if hw_key is not None:
        df_hw_merge = df_hw.copy()

        if hw_key != "node":
            df_hw_merge = df_hw_merge.rename(columns={hw_key: "node"})

        df_hw_merge["node"] = df_hw_merge["node"].astype("string")

        # Eliminar duplicados por nodo
        df_hw_merge = df_hw_merge.drop_duplicates(subset=["node"])

        # Convertir objetos a string para evitar errores
        df_hw_merge = normalize_object_columns(df_hw_merge)

        print("\nHardware listo para unir por 'node':")
        print(df_hw_merge.shape)

    else:
        print("No se encontró columna compatible para unir hardware.")
else:
    print("No existe node_hardware_info.parquet.")

Hardware info cargado:
(343, 19)

Columnas de hardware:
['node', 'cpu_model', 'cpu_core_count_per_socket', 'cpu_threads_per_core', 'cpu_tdp_per_socket', 'cpu_t_case_max', 'cpu_per_core_temp_max', 'cpu_count', 'cpu_core_count_total', 'cpu_memory_bytes_total', 'cpu_tdp_total', 'node_filesystem_size_bytes_total', 'gpu_model', 'gpu_memory_bytes_per_card', 'gpu_tdp_per_card', 'gpu_temp_max', 'gpu_count', 'gpu_memory_bytes_total', 'gpu_tdp_total']

Columnas posibles de ubicación física:
[]

Hardware listo para unir por 'node':
(343, 19)


In [22]:
# ============================================================
# 7. CARGAR SLURM TABLE SOLO SI APORTA COLUMNAS ADICIONALES
# ============================================================

df_slurm_merge = None

if SLURM_PATH.exists():
    slurm_table = pq.read_table(SLURM_PATH)
    df_slurm = clean_decimal_columns_from_table(slurm_table)

    print("SLURM table cargada:")
    print(df_slurm.shape)

    print("\nColumnas de SLURM:")
    print(df_slurm.columns.tolist())

    if "slurm_id" in df_slurm.columns and "slurm_id" in columns_to_read:

        useful_slurm_candidates = [
            "slurm_id",
            "job_name",
            "user",
            "account",
            "partition",
            "qos",
            "submit",
            "start_date",
            "end_date",
            "elapsed",
            "timelimit",
            "state",
            "exit_code",
            "numnodes",
            "numcores",
            "nodetypes",
            "slurm_nodes"
        ]

        useful_slurm_cols = [
            col for col in useful_slurm_candidates
            if col in df_slurm.columns
        ]

        # Agregar solo columnas que no estén ya en prom_slurm_joined
        cols_to_add = [
            col for col in useful_slurm_cols
            if col == "slurm_id" or col not in columns_to_read
        ]

        if len(cols_to_add) > 1:
            df_slurm_merge = df_slurm[cols_to_add].copy()
            df_slurm_merge["slurm_id"] = df_slurm_merge["slurm_id"].astype("string")
            df_slurm_merge = df_slurm_merge.drop_duplicates(subset=["slurm_id"])
            df_slurm_merge = normalize_object_columns(df_slurm_merge)

            print("\nSLURM aportará columnas adicionales:")
            print(cols_to_add)
            print(df_slurm_merge.shape)
        else:
            print("\nSLURM no aporta columnas adicionales importantes.")
    else:
        print("No se puede unir SLURM porque falta slurm_id.")
else:
    print("No existe slurm_table_cleaned.parquet.")

SLURM table cargada:
(1596963, 9)

Columnas de SLURM:
['id', 'start_date', 'end_date', 'node', 'nodetypes', 'numnodes', 'numcores', 'submit', 'state']
No se puede unir SLURM porque falta slurm_id.


In [23]:
# ============================================================
# 8. FUNCIONES PARA ENRIQUECER CADA BLOQUE
# ============================================================

def infer_physical_location(df):
    """
    Intenta inferir rack y posición desde nombres de nodo tipo r10n1.
    Ejemplo:
    r10n1 -> rack_inferred = r10, node_position_inferred = n1
    """
    if "node" not in df.columns:
        return df

    node_str = df["node"].astype(str)

    extracted = node_str.str.extract(r"^(r\d+)(n\d+)$")

    df["rack_inferred"] = extracted[0].astype("string")
    df["node_position_inferred"] = extracted[1].astype("string")

    return df


def enrich_chunk(df):
    """
    Limpia, transforma y enriquece un bloque del dataset grande.
    """

    df = df.copy()

    # Convertir IDs a string para evitar conflictos de tipos
    for col in ["prom_id", "slurm_id", "__index_level_0__"]:
        if col in df.columns:
            df[col] = df[col].astype("string")

    if "node" in df.columns:
        df["node"] = df["node"].astype("string")

    # Columnas numéricas
    numeric_columns = [
        "timestamp_seconds",
        "timestamp_seconds_delta",

        "node_load1",
        "node_load5",
        "node_load15",
        "node_procs_running",
        "node_procs_blocked",
        "node_power_usage",
        "up",

        "node_rapl_package_joules_total_sum",
        "node_rapl_package_joules_total_sum_delta",
        "node_rapl_package_power_sum",

        "node_memory_Active_bytes",
        "node_memory_MemFree_bytes",
        "node_memory_Dirty_bytes",
        "node_memory_MemAvailable_bytes",
        "node_memory_MemTotal_bytes",

        "node_hwmon_temp_celsius_min",
        "node_hwmon_temp_celsius_mean",
        "node_hwmon_temp_celsius_max",
        "node_thermal_zone_temp_min",
        "node_thermal_zone_temp_mean",
        "node_thermal_zone_temp_max",

        "nvidia_gpu_memory_used_bytes_sum",
        "nvidia_gpu_temperature_celsius_0",
        "nvidia_gpu_temperature_celsius_1",
        "nvidia_gpu_temperature_celsius_2",
        "nvidia_gpu_temperature_celsius_3",
        "nvidia_gpu_temperature_celsius_min",
        "nvidia_gpu_temperature_celsius_mean",
        "nvidia_gpu_temperature_celsius_max",

        "nvidia_gpu_fanspeed_percent_min",
        "nvidia_gpu_fanspeed_percent_mean",
        "nvidia_gpu_fanspeed_percent_max",

        "nvidia_gpu_power_usage_milliwatts_0",
        "nvidia_gpu_power_usage_milliwatts_1",
        "nvidia_gpu_power_usage_milliwatts_2",
        "nvidia_gpu_power_usage_milliwatts_3",
        "nvidia_gpu_power_usage_milliwatts_min",
        "nvidia_gpu_power_usage_milliwatts_mean",
        "nvidia_gpu_power_usage_milliwatts_max",
        "nvidia_gpu_power_usage_milliwatts_sum",

        "nvidia_gpu_duty_cycle_min",
        "nvidia_gpu_duty_cycle_mean",
        "nvidia_gpu_duty_cycle_max",

        "node_disk_read_bytes_total_sum_delta",
        "node_disk_written_bytes_total_sum_delta",
        "node_disk_reads_completed_total_sum_delta",
        "node_disk_writes_completed_total_sum_delta",
        "node_disk_io_now_sum",

        "node_network_receive_bytes_total_sum_delta",
        "node_network_transmit_bytes_total_sum_delta",
        "node_network_receive_packets_total_sum_delta",
        "node_network_transmit_packets_total_sum_delta",

        "numnodes",
        "numcores",
        "gpu_node"
    ]

    df = safe_to_numeric(df, numeric_columns)

    # Fechas
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df["date"] = df["timestamp"].dt.date.astype("string")
        df["hour"] = df["timestamp"].dt.hour.astype("float64")
        df["day_of_week"] = df["timestamp"].dt.day_name().astype("string")

    if "start_date" in df.columns:
        df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")

    if "end_date" in df.columns:
        df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce")

    # Duración del job
    if "start_date" in df.columns and "end_date" in df.columns:
        df["job_duration_hours"] = (
            df["end_date"] - df["start_date"]
        ).dt.total_seconds() / 3600

    # RAM en GB
    if "node_memory_Active_bytes" in df.columns:
        df["ram_active_gb"] = df["node_memory_Active_bytes"] / (1024 ** 3)

    if "node_memory_MemFree_bytes" in df.columns:
        df["ram_free_gb"] = df["node_memory_MemFree_bytes"] / (1024 ** 3)

    if "node_memory_MemAvailable_bytes" in df.columns:
        df["ram_available_gb"] = df["node_memory_MemAvailable_bytes"] / (1024 ** 3)

    if "node_memory_MemTotal_bytes" in df.columns:
        df["ram_total_gb"] = df["node_memory_MemTotal_bytes"] / (1024 ** 3)

    if "ram_active_gb" in df.columns and "ram_total_gb" in df.columns:
        df["ram_active_pct"] = (
            df["ram_active_gb"] / df["ram_total_gb"].replace(0, np.nan)
        ) * 100

    # GPU memoria en GB
    if "nvidia_gpu_memory_used_bytes_sum" in df.columns:
        df["gpu_memory_used_gb"] = (
            df["nvidia_gpu_memory_used_bytes_sum"] / (1024 ** 3)
        )

    # GPU potencia en watts
    if "nvidia_gpu_power_usage_milliwatts_mean" in df.columns:
        df["gpu_power_watts_mean"] = (
            df["nvidia_gpu_power_usage_milliwatts_mean"] / 1000
        )

    if "nvidia_gpu_power_usage_milliwatts_max" in df.columns:
        df["gpu_power_watts_max"] = (
            df["nvidia_gpu_power_usage_milliwatts_max"] / 1000
        )

    if "nvidia_gpu_power_usage_milliwatts_sum" in df.columns:
        df["gpu_power_watts_sum"] = (
            df["nvidia_gpu_power_usage_milliwatts_sum"] / 1000
        )

    # Delta de tiempo limpio
    if "timestamp_seconds_delta" in df.columns:
        df["delta_seconds_clean"] = pd.to_numeric(
            df["timestamp_seconds_delta"],
            errors="coerce"
        )

        valid_delta = df["delta_seconds_clean"][
            (df["delta_seconds_clean"] > 0) &
            (df["delta_seconds_clean"] <= 3600)
        ]

        median_delta = valid_delta.median()

        if pd.isna(median_delta):
            median_delta = 30

        df.loc[
            (df["delta_seconds_clean"] <= 0) |
            (df["delta_seconds_clean"] > 3600) |
            (df["delta_seconds_clean"].isna()),
            "delta_seconds_clean"
        ] = median_delta

    # Energía estimada del nodo en kWh
    # Suposición: node_power_usage está en watts
    if "node_power_usage" in df.columns and "delta_seconds_clean" in df.columns:
        df["node_energy_kwh_est"] = (
            df["node_power_usage"] * df["delta_seconds_clean"]
        ) / 3_600_000

    # Energía CPU RAPL en kWh
    if "node_rapl_package_joules_total_sum_delta" in df.columns:
        df["cpu_package_energy_kwh"] = (
            df["node_rapl_package_joules_total_sum_delta"] / 3_600_000
        )

    # Energía GPU estimada en kWh
    if "gpu_power_watts_mean" in df.columns and "delta_seconds_clean" in df.columns:
        df["gpu_energy_kwh_est"] = (
            df["gpu_power_watts_mean"] * df["delta_seconds_clean"]
        ) / 3_600_000

    # CO2 estimado
    CO2_FACTOR_KG_PER_KWH = 0.4

    if "node_energy_kwh_est" in df.columns:
        df["co2_kg_est"] = df["node_energy_kwh_est"] * CO2_FACTOR_KG_PER_KWH

    # Carga por core
    if "node_load1" in df.columns and "numcores" in df.columns:
        df["node_load1_per_core"] = (
            df["node_load1"] / df["numcores"].replace(0, np.nan)
        )

    # Inferir ubicación física desde nombre de nodo
    df = infer_physical_location(df)

    # Enriquecer con hardware
    if df_hw_merge is not None and "node" in df.columns:
        df = df.merge(df_hw_merge, on="node", how="left")

    # Enriquecer con SLURM adicional
    if df_slurm_merge is not None and "slurm_id" in df.columns:
        df = df.merge(df_slurm_merge, on="slurm_id", how="left")

    # Convertir columnas object a string para evitar errores de parquet
    df = normalize_object_columns(df)

    return df

In [24]:
# ============================================================
# 9. CREAR UN SOLO ARCHIVO PARQUET REDUCIDO Y ENRIQUECIDO
# ============================================================

# Si ya existe un archivo anterior, eliminarlo
if OUTPUT_SINGLE_FILE.exists():
    OUTPUT_SINGLE_FILE.unlink()

# Ordenar los archivos de forma aleatoria reproducible para tomar muestra diversa
rng = np.random.default_rng(42)
file_order = list(rng.permutation(parquet_parts))

writer = None
total_rows_written = 0
part_counter = 0

for file_path in file_order:

    print("\nProcesando archivo:")
    print(file_path.name)

    pf = pq.ParquetFile(file_path)

    for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=columns_to_read):

        df_batch = batch.to_pandas()
        df_batch = enrich_chunk(df_batch)

        # Convertir a tabla Arrow
        table = pa.Table.from_pandas(df_batch, preserve_index=False)

        # Crear writer con el esquema del primer bloque
        if writer is None:
            writer = pq.ParquetWriter(
                OUTPUT_SINGLE_FILE,
                table.schema,
                compression="snappy"
            )
            expected_columns = table.schema.names

        else:
            # Asegurar que todos los bloques tengan mismas columnas y orden
            for col in expected_columns:
                if col not in df_batch.columns:
                    df_batch[col] = pd.NA

            df_batch = df_batch[expected_columns]
            df_batch = normalize_object_columns(df_batch)

            table = pa.Table.from_pandas(df_batch, preserve_index=False)

        writer.write_table(table)

        total_rows_written += len(df_batch)
        part_counter += 1

        current_size = OUTPUT_SINGLE_FILE.stat().st_size if OUTPUT_SINGLE_FILE.exists() else 0
        current_size_gb = current_size / (1024 ** 3)

        print(
            f"Bloque {part_counter} | filas acumuladas: {total_rows_written:,} | "
            f"tamaño aproximado: {current_size_gb:.2f} GB"
        )

        if current_size >= TARGET_BYTES:
            print("\nTamaño objetivo alcanzado.")
            break

    if OUTPUT_SINGLE_FILE.exists() and OUTPUT_SINGLE_FILE.stat().st_size >= TARGET_BYTES:
        break

# Cerrar writer
if writer is not None:
    writer.close()

print("\nProceso terminado.")
print("Archivo final:", OUTPUT_SINGLE_FILE)
print("Filas escritas:", total_rows_written)
print("Tamaño final GB:", OUTPUT_SINGLE_FILE.stat().st_size / (1024 ** 3))


Procesando archivo:
part-00152-ed3d2dff-1a97-4ddb-ad80-88573fd4bf4d-c000.snappy.parquet
Bloque 1 | filas acumuladas: 50,000 | tamaño aproximado: 0.00 GB
Bloque 2 | filas acumuladas: 100,000 | tamaño aproximado: 0.01 GB
Bloque 3 | filas acumuladas: 150,000 | tamaño aproximado: 0.01 GB
Bloque 4 | filas acumuladas: 200,000 | tamaño aproximado: 0.02 GB
Bloque 5 | filas acumuladas: 250,000 | tamaño aproximado: 0.02 GB
Bloque 6 | filas acumuladas: 300,000 | tamaño aproximado: 0.03 GB
Bloque 7 | filas acumuladas: 350,000 | tamaño aproximado: 0.03 GB
Bloque 8 | filas acumuladas: 400,000 | tamaño aproximado: 0.04 GB
Bloque 9 | filas acumuladas: 450,000 | tamaño aproximado: 0.04 GB
Bloque 10 | filas acumuladas: 466,914 | tamaño aproximado: 0.04 GB

Procesando archivo:
part-00026-ed3d2dff-1a97-4ddb-ad80-88573fd4bf4d-c000.snappy.parquet
Bloque 11 | filas acumuladas: 516,914 | tamaño aproximado: 0.05 GB
Bloque 12 | filas acumuladas: 566,914 | tamaño aproximado: 0.05 GB
Bloque 13 | filas acumuladas

In [25]:
# ============================================================
# 10. VALIDAR EL ARCHIVO FINAL SIN CARGARLO COMPLETO
# ============================================================

pf_final = pq.ParquetFile(OUTPUT_SINGLE_FILE)

print("Archivo final:")
print(OUTPUT_SINGLE_FILE)

print("\nNúmero de filas:")
print(pf_final.metadata.num_rows)

print("\nNúmero de columnas:")
print(len(pf_final.schema.names))

print("\nTamaño GB:")
print(OUTPUT_SINGLE_FILE.stat().st_size / (1024 ** 3))

print("\nColumnas finales:")
for col in pf_final.schema.names:
    print(col)

Archivo final:
D:\Descargas\datacenter_enriched_1gb_single.parquet

Número de filas:
11930727

Número de columnas:
101

Tamaño GB:
1.0044815642759204

Columnas finales:
prom_id
timestamp
timestamp_seconds
timestamp_seconds_delta
node
gpu_node
node_load1
node_load5
node_load15
node_procs_running
node_procs_blocked
node_power_usage
up
node_rapl_package_joules_total_sum
node_rapl_package_joules_total_sum_delta
node_rapl_package_power_sum
node_memory_Active_bytes
node_memory_MemFree_bytes
node_memory_Dirty_bytes
node_hwmon_temp_celsius_min
node_hwmon_temp_celsius_mean
node_hwmon_temp_celsius_max
node_thermal_zone_temp_min
node_thermal_zone_temp_mean
node_thermal_zone_temp_max
nvidia_gpu_memory_used_bytes_sum
nvidia_gpu_temperature_celsius_0
nvidia_gpu_temperature_celsius_1
nvidia_gpu_temperature_celsius_2
nvidia_gpu_temperature_celsius_3
nvidia_gpu_temperature_celsius_min
nvidia_gpu_temperature_celsius_mean
nvidia_gpu_temperature_celsius_max
nvidia_gpu_fanspeed_percent_min
nvidia_gpu_fansp

In [26]:
# ============================================================
# 11. LEER SOLO UNA MUESTRA DEL ARCHIVO FINAL
# ============================================================

# Leer solo las primeras 50.000 filas para validar
sample_batches = []

for batch in pf_final.iter_batches(batch_size=50_000):
    sample_batches.append(batch.to_pandas())
    break

df_sample_final = pd.concat(sample_batches, ignore_index=True)

print("Muestra cargada:")
print(df_sample_final.shape)

display(df_sample_final.head())

print("\nRango temporal de la muestra:")
print(df_sample_final["timestamp"].min(), "→", df_sample_final["timestamp"].max())

print("\nNodos únicos en la muestra:")
print(df_sample_final["node"].nunique())

if "gpu_node" in df_sample_final.columns:
    print("\nDistribución CPU/GPU en la muestra:")
    print(df_sample_final["gpu_node"].value_counts(dropna=False))

if "state" in df_sample_final.columns:
    print("\nEstados SLURM en la muestra:")
    print(df_sample_final["state"].value_counts(dropna=False).head(20))

Muestra cargada:
(50000, 101)


,prom_id,timestamp,timestamp_seconds,timestamp_seconds_delta,node,gpu_node,node_load1,node_load5,node_load15,node_procs_running,...,cpu_memory_bytes_total,cpu_tdp_total,node_filesystem_size_bytes_total,gpu_model,gpu_memory_bytes_per_card,gpu_tdp_per_card,gpu_temp_max,gpu_count,gpu_memory_bytes_total,gpu_tdp_total
0,21988367,2022-07-28 22:58:30,1659049110,30.0,r26n7,0,3.07,3.66,3.72,9.0,...,1.030792e+11,170.0,1.992225e+12,n/a,0.0,0.0,0.0,0,0.0,0.0
1,21988368,2022-07-28 22:59:00,1659049140,30.0,r26n7,0,3.04,3.60,3.69,17.0,...,1.030792e+11,170.0,1.992225e+12,n/a,0.0,0.0,0.0,0,0.0,0.0
2,21988369,2022-07-28 22:59:30,1659049170,30.0,r26n7,0,3.02,3.54,3.67,14.0,...,1.030792e+11,170.0,1.992225e+12,n/a,0.0,0.0,0.0,0,0.0,0.0
3,21988370,2022-07-28 23:00:00,1659049200,30.0,r26n7,0,3.01,3.49,3.64,19.0,...,1.030792e+11,170.0,1.992225e+12,n/a,0.0,0.0,0.0,0,0.0,0.0
4,21988371,2022-07-28 23:00:30,1659049230,30.0,r26n7,0,3.01,3.44,3.62,17.0,...,1.030792e+11,170.0,1.992225e+12,n/a,0.0,0.0,0.0,0,0.0,0.0



Rango temporal de la muestra:
2022-07-28 22:58:30 → 2022-08-16 02:37:00

Nodos únicos en la muestra:
1

Distribución CPU/GPU en la muestra:
gpu_node
0    50000
Name: count, dtype: int64

Estados SLURM en la muestra:
state
COMPLETED    35022
TIMEOUT      12651
CANCELLED     2319
FAILED           8
Name: count, dtype: Int64


In [ ]:
# REducir columnas a un subconjunto más pequeño para análisis rápido
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

INPUT_FILE = Path(r"D:\Descargas\datacenter_enriched_1gb_single.parquet")
OUTPUT_CORE_FILE = Path(r"D:\Descargas\datacenter_enriched_core_single.parquet")

pf = pq.ParquetFile(INPUT_FILE)
available_cols = pf.schema.names

core_columns = [
    "prom_id",
    "timestamp",
    "timestamp_seconds",
    "timestamp_seconds_delta",
    "node",
    "rack_inferred",
    "node_position_inferred",

    "gpu_node",
    "gpu_model",
    "gpu_count",
    "cpu_tdp_total",
    "gpu_tdp_total",
    "filesystem_size_bytes_total",

    "node_load1",
    "node_load5",
    "node_load15",
    "node_load1_per_core",
    "node_power_usage",
    "node_rapl_package_power_sum",
    "node_rapl_package_joules_total_sum_delta",
    "cpu_package_energy_kwh",

    "node_memory_Active_bytes",
    "node_memory_MemFree_bytes",
    "node_memory_MemTotal_bytes",
    "ram_active_gb",
    "ram_free_gb",
    "ram_total_gb",
    "ram_active_pct",

    "node_hwmon_temp_celsius_mean",
    "node_hwmon_temp_celsius_max",
    "node_thermal_zone_temp_mean",
    "node_thermal_zone_temp_max",

    "nvidia_gpu_memory_used_bytes_sum",
    "gpu_memory_used_gb",
    "nvidia_gpu_temperature_celsius_mean",
    "nvidia_gpu_temperature_celsius_max",
    "nvidia_gpu_power_usage_milliwatts_mean",
    "gpu_power_watts_mean",
    "gpu_power_watts_max",
    "nvidia_gpu_duty_cycle_mean",
    "nvidia_gpu_duty_cycle_max",

    "node_disk_read_bytes_total_sum_delta",
    "node_disk_written_bytes_total_sum_delta",
    "node_network_receive_bytes_total_sum_delta",
    "node_network_transmit_bytes_total_sum_delta",

    "node_energy_kwh_est",
    "gpu_energy_kwh_est",
    "co2_kg_est",

    "slurm_id",
    "start_date",
    "end_date",
    "job_duration_hours",
    "state",
    "nodetypes",
    "numnodes",
    "numcores",
    "slurm_nodes"
]

# Mantener solo columnas existentes
final_columns = [col for col in core_columns if col in available_cols]

print("Columnas disponibles en el archivo original:", len(available_cols))
print("Columnas que se conservarán:", len(final_columns))

for col in final_columns:
    print(col)

# Leer solo esas columnas
df_core = pd.read_parquet(INPUT_FILE, columns=final_columns)

print("Dataset limpio:")
print(df_core.shape)

display(df_core.head())

# Guardar versión limpia
df_core.to_parquet(OUTPUT_CORE_FILE, index=False)

print("Archivo limpio guardado en:")
print(OUTPUT_CORE_FILE)

print("Tamaño aproximado MB:")
print(OUTPUT_CORE_FILE.stat().st_size / (1024 ** 2))

Columnas disponibles en el archivo original: 101
Columnas que se conservarán: 53
prom_id
timestamp
timestamp_seconds
timestamp_seconds_delta
node
rack_inferred
node_position_inferred
gpu_node
gpu_model
gpu_count
cpu_tdp_total
gpu_tdp_total
node_load1
node_load5
node_load15
node_load1_per_core
node_power_usage
node_rapl_package_power_sum
node_rapl_package_joules_total_sum_delta
cpu_package_energy_kwh
node_memory_Active_bytes
node_memory_MemFree_bytes
ram_active_gb
ram_free_gb
node_hwmon_temp_celsius_mean
node_hwmon_temp_celsius_max
node_thermal_zone_temp_mean
node_thermal_zone_temp_max
nvidia_gpu_memory_used_bytes_sum
gpu_memory_used_gb
nvidia_gpu_temperature_celsius_mean
nvidia_gpu_temperature_celsius_max
nvidia_gpu_power_usage_milliwatts_mean
gpu_power_watts_mean
gpu_power_watts_max
nvidia_gpu_duty_cycle_mean
nvidia_gpu_duty_cycle_max
node_disk_read_bytes_total_sum_delta
node_disk_written_bytes_total_sum_delta
node_network_receive_bytes_total_sum_delta
node_network_transmit_bytes_tota

,prom_id,timestamp,timestamp_seconds,timestamp_seconds_delta,node,rack_inferred,node_position_inferred,gpu_node,gpu_model,gpu_count,...,co2_kg_est,slurm_id,start_date,end_date,job_duration_hours,state,nodetypes,numnodes,numcores,slurm_nodes
0,21988367,2022-07-28 22:58:30,1659049110,30.0,r26n7,r26,n7,0,n/a,0,...,0.000387,1740264,2022-07-28 05:00:01,2022-07-29 17:25:42,36.428056,COMPLETED,normal(1),1,16,['r26n7']
1,21988368,2022-07-28 22:59:00,1659049140,30.0,r26n7,r26,n7,0,n/a,0,...,0.000387,1740264,2022-07-28 05:00:01,2022-07-29 17:25:42,36.428056,COMPLETED,normal(1),1,16,['r26n7']
2,21988369,2022-07-28 22:59:30,1659049170,30.0,r26n7,r26,n7,0,n/a,0,...,0.000387,1740264,2022-07-28 05:00:01,2022-07-29 17:25:42,36.428056,COMPLETED,normal(1),1,16,['r26n7']
3,21988370,2022-07-28 23:00:00,1659049200,30.0,r26n7,r26,n7,0,n/a,0,...,0.000387,1740264,2022-07-28 05:00:01,2022-07-29 17:25:42,36.428056,COMPLETED,normal(1),1,16,['r26n7']
4,21988371,2022-07-28 23:00:30,1659049230,30.0,r26n7,r26,n7,0,n/a,0,...,0.000387,1740264,2022-07-28 05:00:01,2022-07-29 17:25:42,36.428056,COMPLETED,normal(1),1,16,['r26n7']


Archivo limpio guardado en:
D:\Descargas\datacenter_enriched_core_single.parquet
Tamaño aproximado MB:
716.4406881332397
